In [ ]:
!git clone https://github.com/olivkoch/TinyRecursiveModels.git /content/TinyRecursiveModels
!uv pip install --system -e /content/TinyRecursiveModels

Cloning into '/content/TinyRecursiveModels'...
remote: Enumerating objects: 123, done.
remote: Total 123 (delta 0), reused 0 (delta 0), pack-reused 123 (from 1)
Receiving objects: 100% (123/123), 1.28 MiB | 3.93 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Using Python 3.12.13 environment at: /usr
Resolved 113 packages in 1.10s
Prepared 14 packages in 1.26s
Uninstalled 1 package in 4ms
Installed 14 packages in 25ms
 + argdantic==1.3.3
 + colorama==0.4.6
 + colorlog==6.10.1
 + coolname==5.0.0
 + hydra-colorlog==1.2.0
 + hydra-core==1.3.2
 + lightning==2.6.5
 + lightning-utilities==0.15.3
 + official-trm==0.1.0 (from file:///content/TinyRecursiveModels)
 + pytorch-lightning==2.6.5
 - rich==13.9.4
 + rich==15.0.0
 + taskipy==1.14.1
 + tomli==2.4.1
 + torchmetrics==1.9.0


In [ ]:
import sys
sys.path.insert(0, "/content/TinyRecursiveModels")

In [ ]:
import sys
import os
import numpy as np
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

import torch
import yaml

from typing import Any
from pathlib import Path
from pretrain import PretrainConfig
from utils.functions import load_model_class

from torch.utils.data import TensorDataset, DataLoader

sys.path.insert(0, "/content/TinyRecursiveModels")

abs_config_dir = os.path.abspath("./TinyRecursiveModels/config")

with initialize_config_dir(version_base=None, config_dir=abs_config_dir):
    cfg = compose(config_name="cfg_pretrain")
    config_dict = OmegaConf.to_container(cfg, resolve=True)

Failed to import AdamATan2. Using AdamW instead


In [ ]:
def get_maze_dataset(
    dataset_path: str | Path,
    split: str = "train",
    n_puzzles: int = 5,
    seed: int = 42,
    set_name: str = "all",
):
    rng = np.random.default_rng(seed)
    split_dir = Path(dataset_path) / split

    inputs = np.load(split_dir / f"{set_name}__inputs.npy", mmap_mode="r")
    labels = np.load(split_dir / f"{set_name}__labels.npy", mmap_mode="r")
    puzzle_ids = np.load(split_dir / f"{set_name}__puzzle_identifiers.npy")

    n = len(inputs)
    n_puzzles = min(n_puzzles, n)

    idx = rng.choice(n, size=n_puzzles, replace=False)

    return TensorDataset(
        torch.from_numpy(inputs[idx].astype(np.float32)),
        torch.from_numpy(labels[idx].astype(np.int64)),
        torch.from_numpy(puzzle_ids[idx].astype(np.int64)),
    )


def get_aug_dataset(
    dataset_path: str | Path,
    split: str = "train",
    n_puzzles: int = 5,
    seed: int = 42,
    set_name: str = "all",
):
    rng = np.random.default_rng(seed)
    split_dir = Path(dataset_path) / split

    inputs = np.load(split_dir / f"{set_name}__inputs.npy", mmap_mode="r")
    labels = np.load(split_dir / f"{set_name}__labels.npy", mmap_mode="r")
    puzzle_ids = np.load(split_dir / f"{set_name}__puzzle_identifiers.npy")
    puzzle_indices = np.load(split_dir / f"{set_name}__puzzle_indices.npy")

    total_puzzles = len(puzzle_indices) - 1
    n_puzzles = min(n_puzzles, total_puzzles)

    chosen_puzzles = rng.choice(total_puzzles, size=n_puzzles, replace=False)

    xs, ys, ids = [], [], []

    for p in chosen_puzzles:
        start = puzzle_indices[p]
        end = puzzle_indices[p + 1]

        ex = rng.integers(start, end)

        xs.append(inputs[ex])
        ys.append(labels[ex])
        ids.append(puzzle_ids[p])  # puzzle-level id corretto

    return TensorDataset(
        torch.from_numpy(np.stack(xs).astype(np.float32)),
        torch.from_numpy(np.stack(ys).astype(np.int64)),
        torch.from_numpy(np.array(ids).astype(np.int64)),
    )



def instantiate_model(
    config: PretrainConfig,
    global_batch_size: int,
    vocab_size: int,
    seq_len: int,
    num_puzzle_identifiers: int,
    device: str,
):
    model_cfg = dict(
        **config.arch.__pydantic_extra__,
        batch_size=global_batch_size,
        vocab_size=vocab_size,
        seq_len=seq_len,
        num_puzzle_identifiers=num_puzzle_identifiers,
        causal=False,
    )

    model_cls = load_model_class(config.arch.name)
    loss_head_cls = load_model_class(config.arch.loss.name)

    print(model_cfg)

    with torch.device(device):
        base_model = model_cls(model_cfg)

        model = loss_head_cls(base_model, **config.arch.loss.__pydantic_extra__)

    model.to(device)
    model.eval()
    return model


def load_trm(
    checkpoint_file: str,
    config_dict: dict[str, Any],
    global_batch_size: int,
    vocab_size: int,
    seq_len: int,
    num_puzzle_identifiers: int,
    device: str = "cpu",
) :
    config = PretrainConfig(**config_dict)

    torch_device = torch.device(device)

    model = instantiate_model(
        config,
        global_batch_size,
        vocab_size,
        seq_len,
        num_puzzle_identifiers,
        device
    )

    state_dict = torch.load(checkpoint_file, map_location=torch_device)

    puzzle_emb_name = "_orig_mod.model.inner.puzzle_emb.weights"
    if hasattr(model, "model") and hasattr(model.model, "puzzle_emb"):
        expected_shape: torch.Size = model.model.puzzle_emb.weights.shape  # type: ignore[attr-defined]
        print('Expected shape', expected_shape)
        if puzzle_emb_name in state_dict:
            puzzle_emb = state_dict[puzzle_emb_name]
            if puzzle_emb.shape != expected_shape:
                state_dict[puzzle_emb_name] = (
                    torch.mean(puzzle_emb, dim=0, keepdim=True)
                    .expand(expected_shape)
                    .contiguous()
                )

    model.load_state_dict(state_dict, strict=False)
    model.to(torch_device)
    model.eval()

    return model


@torch.inference_mode()
def collect_trajectory(model, batch, N_sup: int):
  with torch.device('cuda'):
    carry = model.initial_carry(batch)

  B, T, S = carry.inner_carry.z_H.shape
  traj = np.zeros((N_sup, B, T, S), dtype=np.float32)

  for step in range(N_sup):
      carry, _, _, _, _ = model(
          carry=carry,
          batch=batch,
          return_keys=[]
      )
      traj[step] = carry.inner_carry.z_H.float().cpu().numpy()

  return traj


def collect_all_trajectories(model, loader, N_sup: int, traj_dir: Path):
  traj_dir.mkdir(parents=True, exist_ok=True)

  processed_items = 0
  set_name = 'all'

  processed_batches = 0

  for inputs, labels, puzzle_identifiers in loader:
      processed_items += inputs.shape[0]
      processed_batches += 1

      print(f'Processing batch {processed_batches}')

      batch = {
          "inputs": inputs.to('cuda'),
          "labels": labels.to('cuda'),
          "puzzle_identifiers": puzzle_identifiers.to('cuda')
      }

      traj = collect_trajectory(model, batch, N_sup)

      np.savez(traj_dir / f'batch_{processed_batches}.npz', y_trajectories=traj)

  print(f'Processed: {processed_items} items, {processed_batches} batches')

In [ ]:
def maze30x30(split: str, puzzles: int):
  global_batch_size = 64

  model = load_trm(
      checkpoint_file="/content/drive/MyDrive/shared/Maze30x30/maze_hard_step_32550",
      config_dict=config_dict,
      global_batch_size=global_batch_size,
      vocab_size=6,
      seq_len=900,
      num_puzzle_identifiers=1,
      device='cuda'
  )

  loader = DataLoader(
      dataset=get_maze_dataset(
          dataset_path="/content/drive/MyDrive/shared/Maze30x30/maze-30x30-hard-1k",
          split=split,
          n_puzzles=puzzles,
          seed=42
      ),
      batch_size=global_batch_size,
      num_workers=1,
      prefetch_factor=2
  )

  collect_all_trajectories(
      model,
      loader,
      N_sup=16,
      traj_dir=Path(f'/content/drive/MyDrive/maze30x30_{split}_trajectories')
  )


maze30x30('train', 256)
maze30x30('test', 256)

{'halt_exploration_prob': 0.1, 'halt_max_steps': 16, 'H_cycles': 3, 'L_cycles': 6, 'H_layers': 0, 'L_layers': 2, 'hidden_size': 512, 'num_heads': 8, 'expansion': 4, 'puzzle_emb_ndim': 512, 'pos_encodings': 'rope', 'forward_dtype': 'bfloat16', 'mlp_t': False, 'puzzle_emb_len': 16, 'no_ACT_continue': True, 'batch_size': 64, 'vocab_size': 6, 'seq_len': 900, 'num_puzzle_identifiers': 1, 'causal': False}
Expected shape torch.Size([1, 512])
Inputs shape torch.Size([64, 900])
labels shape torch.Size([64, 900])
puzzle_identifiers shape torch.Size([64])
Processing batch 1
Inputs shape torch.Size([64, 900])
labels shape torch.Size([64, 900])
puzzle_identifiers shape torch.Size([64])
Processing batch 2
Inputs shape torch.Size([64, 900])
labels shape torch.Size([64, 900])
puzzle_identifiers shape torch.Size([64])
Processing batch 3
Inputs shape torch.Size([64, 900])
labels shape torch.Size([64, 900])
puzzle_identifiers shape torch.Size([64])
Processing batch 4
Processed: 256 items, 4 batches
{'hal

In [ ]:
def arcv1(split: str, puzzles: int):
  global_batch_size = 64

  model = load_trm(
      checkpoint_file="/content/drive/MyDrive/shared/ARC-AGI/checkpoints/arc-agi_v1/checkpoint",
      config_dict=config_dict,
      global_batch_size=global_batch_size,
      vocab_size=12,
      seq_len=900,
      num_puzzle_identifiers=876406,
      device='cuda'
  )

  loader = DataLoader(
      dataset=get_aug_dataset(
          dataset_path="/content/drive/MyDrive/shared/ARC-AGI/arc1concept-aug-1000",
          split=split,
          n_puzzles=puzzles,
          seed=42
      ),
      batch_size=global_batch_size,
      num_workers=1,
      prefetch_factor=2
  )

  collect_all_trajectories(
      model,
      loader,
      N_sup=16,
      traj_dir=Path(f'/content/drive/MyDrive/agi1_{split}_trajectories')
  )

arcv1('train', 256)
arcv1('test', 256)


{'halt_exploration_prob': 0.1, 'halt_max_steps': 16, 'H_cycles': 3, 'L_cycles': 6, 'H_layers': 0, 'L_layers': 2, 'hidden_size': 512, 'num_heads': 8, 'expansion': 4, 'puzzle_emb_ndim': 512, 'pos_encodings': 'rope', 'forward_dtype': 'bfloat16', 'mlp_t': False, 'puzzle_emb_len': 16, 'no_ACT_continue': True, 'batch_size': 64, 'vocab_size': 12, 'seq_len': 900, 'num_puzzle_identifiers': 876406, 'causal': False}
Expected shape torch.Size([876406, 512])
Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processed: 256 items, 4 batches
{'halt_exploration_prob': 0.1, 'halt_max_steps': 16, 'H_cycles': 3, 'L_cycles': 6, 'H_layers': 0, 'L_layers': 2, 'hidden_size': 512, 'num_heads': 8, 'expansion': 4, 'puzzle_emb_ndim': 512, 'pos_encodings': 'rope', 'forward_dtype': 'bfloat16', 'mlp_t': False, 'puzzle_emb_len': 16, 'no_ACT_continue': True, 'batch_size': 64, 'vocab_size': 12, 'seq_len': 900, 'num_puzzle_identifiers': 876406, 'causal': False}
Expected shape torch.Size([876406

In [ ]:
def arcv2(split: str, puzzles: int):
  global_batch_size = 64

  model = load_trm(
      checkpoint_file="/content/drive/MyDrive/shared/ARC-AGI/checkpoints/arc-agi_v2/checkpoint",
      config_dict=config_dict,
      global_batch_size=global_batch_size,
      vocab_size=12,
      seq_len=900,
      num_puzzle_identifiers=1191730,
      device='cuda'
  )

  print('Loaded model')

  loader = DataLoader(
      dataset=get_aug_dataset(
          dataset_path="/content/drive/MyDrive/shared/ARC-AGI/arc2concept-aug-1000",
          split=split,
          n_puzzles=puzzles,
          seed=42
      ),
      batch_size=global_batch_size,
      num_workers=1,
      prefetch_factor=2
  )

  print('Loaded data')

  collect_all_trajectories(
      model,
      loader,
      N_sup=16,
      traj_dir=Path(f'/content/drive/MyDrive/agi2_{split}_trajectories')
  )

arcv2('train', 256)
arcv2('test', 256)


{'halt_exploration_prob': 0.1, 'halt_max_steps': 16, 'H_cycles': 3, 'L_cycles': 6, 'H_layers': 0, 'L_layers': 2, 'hidden_size': 512, 'num_heads': 8, 'expansion': 4, 'puzzle_emb_ndim': 512, 'pos_encodings': 'rope', 'forward_dtype': 'bfloat16', 'mlp_t': False, 'puzzle_emb_len': 16, 'no_ACT_continue': True, 'batch_size': 64, 'vocab_size': 12, 'seq_len': 900, 'num_puzzle_identifiers': 1191730, 'causal': False}
Expected shape torch.Size([1191730, 512])
Loaded model
Loaded data
Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processed: 256 items, 4 batches
{'halt_exploration_prob': 0.1, 'halt_max_steps': 16, 'H_cycles': 3, 'L_cycles': 6, 'H_layers': 0, 'L_layers': 2, 'hidden_size': 512, 'num_heads': 8, 'expansion': 4, 'puzzle_emb_ndim': 512, 'pos_encodings': 'rope', 'forward_dtype': 'bfloat16', 'mlp_t': False, 'puzzle_emb_len': 16, 'no_ACT_continue': True, 'batch_size': 64, 'vocab_size': 12, 'seq_len': 900, 'num_puzzle_identifiers': 1191730, 'causal': False}
Expec